# recipe-dataclass composite — cx7: build a Recipe at forward time — unbox args + parents dict + freeze recipe

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `recipe-dataclass`, `parents-dict-by-argidx`, `unbox-args-tensor-to-array`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "recipe-dataclass"
DD_ATOM_IDS = ["recipe-dataclass", "parents-dict-by-argidx", "unbox-args-tensor-to-array"]
DD_SUBTOPICS = ["Backprop: Recipe dataclass", "Backprop: Parents dict by argidx", "Backprop: Unbox Tensor args to array"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing three atoms into the forward half of wrap_forward_fn

The wrapper's forward half does three things in one pass:

1. **Unbox** every `MiniTensor` positional arg to its raw `.array` (so `fwd_fn` sees plain `torch.Tensor`).
2. **Build parents** — a `{argidx: MiniTensor}` dict pulled from the ORIGINAL args (keeping the original positional index).
3. **Freeze the Recipe** — `Recipe(fwd_fn, raw_args, kwargs, parents)` in that order.

All three share the same `isinstance(a, MiniTensor)` scan over `args`. Doing them in one pass is the canonical pattern. This drill makes you wire all three correctly together.

### Composite Exercise — build a Recipe at forward time — unbox args + parents dict + freeze recipe

**Atoms exercised together**: `recipe-dataclass`, `parents-dict-by-argidx`, `unbox-args-tensor-to-array`

Implement `cx7_make_recipe(fwd_fn, args, kwargs)` — the forward half of `wrap_forward_fn`. Given a raw `fwd_fn` (e.g. `t.log`), a tuple `args` (possibly mixed `MiniTensor` + scalars), and a `kwargs` dict, return a `(raw_out, recipe)` pair where:

- `raw_out = fwd_fn(*unbox_args(args), **kwargs)` — the raw `torch.Tensor` output.
- `recipe = Recipe(fwd_fn, unboxed_args, kwargs, parents)` with EXACTLY four fields in that order.
- `unboxed_args` is `args` with every `MiniTensor` replaced by its `.array` (non-Tensors passed through, order preserved).
- `parents = {argidx: a for argidx, a in enumerate(args) if isinstance(a, MiniTensor)}` — filter out non-Tensors but KEEP the original positional index.

Identity matters: `recipe.args[i]` for any MiniTensor input MUST be the same object as `args[i].array` (no copy). `parents[idx]` MUST be the same MiniTensor object.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx7_make_recipe(fwd_fn, args, kwargs):
    """Return (raw_out, recipe). Recipe stores raw args + parents dict."""
    raise NotImplementedError

def _test_cx7():
    from dataclasses import fields
    # --- shape: log_forward(x) — single Tensor arg ---
    x = MiniTensor(t.tensor([1.0, t.e, t.e * t.e]))
    raw_out, recipe = cx7_make_recipe(t.log, (x,), {})
    assert t.allclose(raw_out, t.tensor([0.0, 1.0, 2.0]), atol=1e-5), 'log output wrong'
    assert [f.name for f in fields(Recipe)] == ['func','args','kwargs','parents'], 'Recipe field order changed'
    assert recipe.func is t.log
    assert recipe.args == (x.array,)
    assert recipe.args[0] is x.array, 'recipe.args[0] must BE x.array (identity, not copy)'
    assert recipe.kwargs == {}
    assert recipe.parents == {0: x}
    assert recipe.parents[0] is x, 'parents[0] must be the SAME MiniTensor object'
    # --- mixed: multiply(x, 3.0) — scalar must skip parents, stay in args ---
    x = MiniTensor(t.tensor([2.0, 4.0]))
    raw_out, recipe = cx7_make_recipe(t.multiply, (x, 3.0), {})
    assert t.allclose(raw_out, t.tensor([6.0, 12.0]))
    assert recipe.args == (x.array, 3.0), f'args must keep float in position 1: {recipe.args}'
    assert recipe.parents == {0: x}, f'float at arg-1 must be skipped from parents: {recipe.parents}'
    # --- two MiniTensors at non-adjacent positions: float, T, T ---
    a = MiniTensor(t.tensor([1.0]))
    b = MiniTensor(t.tensor([2.0]))
    def _add_three(s, x, y): return s * (x + y)
    raw_out, recipe = cx7_make_recipe(_add_three, (0.5, a, b), {})
    assert t.allclose(raw_out, t.tensor([1.5]))
    assert recipe.args == (0.5, a.array, b.array)
    assert recipe.parents == {1: a, 2: b}, 'parents must KEEP original argidx (1, 2 — not 0, 1)'
    # --- kwargs preserved verbatim ---
    x = MiniTensor(t.ones(3, 4))
    raw_out, recipe = cx7_make_recipe(t.sum, (x,), {'dim': 1})
    assert recipe.kwargs == {'dim': 1}
    assert t.allclose(raw_out, t.full((3,), 4.0))
    # --- raw torch.Tensor in args must be SKIPPED from parents but pass through ---
    raw_pass = t.tensor([7.0])
    x = MiniTensor(t.tensor([3.0]))
    raw_out, recipe = cx7_make_recipe(t.multiply, (raw_pass, x), {})
    assert recipe.args == (raw_pass, x.array)
    assert recipe.args[0] is raw_pass, 'raw torch.Tensor passes through untouched'
    assert recipe.parents == {1: x}, 'only MiniTensor counts as a parent'
    _dd_passed.add('cx7')

_test_cx7()

<details><summary>Show solution — cx7</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx7_make_recipe(fwd_fn, args, kwargs):
    # one pass over args: same isinstance gate, three transforms.
    raw_args = tuple(
        a.array if isinstance(a, MiniTensor) else a
        for a in args
    )
    parents = {
        idx: a
        for idx, a in enumerate(args)
        if isinstance(a, MiniTensor)
    }
    raw_out = fwd_fn(*raw_args, **kwargs)
    recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return raw_out, recipe
```

**Why one pass.** Both `unbox` and `parents` share the same `isinstance(a, MiniTensor)` predicate over the same `args` tuple — you could inline them into a single loop with two builders. The comprehensions above keep the intent crisp at the cost of one extra scan; either is fine.

**Why the recipe stores `raw_args`, not `args`.** Reverse pass replays the forward call with `fwd_fn(*recipe.args, **recipe.kwargs)` — that needs the unboxed view, not the MiniTensor wrappers. The MiniTensors live on in `parents` so the reverse traversal can find them.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["Backprop: Recipe dataclass", "Backprop: Parents dict by argidx", "Backprop: Unbox Tensor args to array"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()